In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# ============================================================
# 2. LOAD DATA
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

train = pd.read_csv(DATA_DIR / "train.csv",index_col=0)
test = pd.read_csv(DATA_DIR / "test.csv",index_col=0)

print("Training shape:", train.shape)
print("Testing shape:", test.shape)

Training shape: (103904, 24)
Testing shape: (25976, 24)


# <font color = "Orange"> Plan </font>

## Build 2 models:
### - Passenger and flight characteristics only model <font color = "Orange"> (model A) </font>: Age, Gender, Customer Type, Type of Travel, Class, Flight Distance, Departure and Arrival Delays
### - Full passenger experience model <font color = "Orange"> (model B) </font>: Everything in A and Online Boarding, Inflight Entertainment, Seat Comfort, Wifi, Other ratings

# <font color = "Orange">  1. Preparing data </font>

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Define target
target = "satisfaction"

# Separate features and target
X = train.drop(columns=[target, "id"])
y = train[target].map({
    "neutral or dissatisfied": 0,
    "satisfied": 1
})

# Remove identifier from test features too
X_test = test.drop(columns=["id"])

# Stratified train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check distributions
print("Training:", y_train.value_counts(normalize=True))
print("Validation:", y_val.value_counts(normalize=True))
print("Kaggle test:", test[target].value_counts(normalize=True))

Training: satisfaction
0    0.566666
1    0.433334
Name: proportion, dtype: float64
Validation: satisfaction
0    0.566671
1    0.433329
Name: proportion, dtype: float64
Kaggle test: satisfaction
neutral or dissatisfied    0.561018
satisfied                  0.438982
Name: proportion, dtype: float64


# <font color = "Orange">  2. Building a Logistic Regression baseline </font>

### Logistic Regression cannot interpret strings, need to convert into numerical representations (via one-hot encoding), also need to standardize numerical features

## 2.1 Preprocessing

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

# ============================================================
# DEFINE FEATURE GROUPS
# ============================================================

categorical_features = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

numerical_features = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

# Wi-Fi is handled separately because 0 = Not Applicable
categorical_features_model_b = categorical_features + [
    "Inflight wifi service"
]

# ============================================================
# NUMERICAL PREPROCESSING
# ============================================================

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# ============================================================
# CATEGORICAL PREPROCESSING
# ============================================================

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ============================================================
# MODEL A PREPROCESSING
# ============================================================

preprocessor_a = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# ============================================================
# MODEL B PREPROCESSING
# ============================================================

preprocessor_b = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features_model_b),
    ("service", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), [
        "Online boarding",
        "Inflight entertainment",
        "Seat comfort",
        "On-board service",
        "Leg room service",
        "Cleanliness",
        "Checkin service",
        "Baggage handling",
        "Inflight service",
        "Food and drink",
        "Ease of Online booking",
        "Gate location",
        "Departure/Arrival time convenient"
    ])
])

## 2.2 Train and evaluate Logistic Regression baseline

In [ ]:
# Model A

model_a = Pipeline(steps=[
    ("preprocessor", preprocessor_a),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Model B
model_b = Pipeline(steps=[
    ("preprocessor", preprocessor_b),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train models
model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)

print("Model A trained.")
print("Model B trained.")

Model A trained.
Model B trained.


In [8]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)
import matplotlib.pyplot as plt

y_pred = model_a.predict(X_val)
y_proba = model_a.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_proba))

print("\nClassification report:")
print(classification_report(y_val, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_pred))



Accuracy: 0.7863914152350705
ROC-AUC: 0.847368418322852

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.80      0.81     11776
           1       0.74      0.77      0.76      9005

    accuracy                           0.79     20781
   macro avg       0.78      0.79      0.78     20781
weighted avg       0.79      0.79      0.79     20781


Confusion matrix:
[[9366 2410]
 [2029 6976]]
